In [0]:
import csv, os
from datetime import date
from decimal import Decimal
from pyspark.sql import functions as F

# SIMULATION ONLY: builds a fake finance billing feed from real delivered spend,
# with documented, deliberately planted discrepancies for the reconciliation to find.

delivered = (spark.table("workspace.media_pipeline.silver_campaign_performance")
    .withColumn("billing_month",
                F.coalesce(F.date_format("report_date", "yyyy-MM"), F.lit("UNDATED")))
    .groupBy("platform", "billing_month")
    .agg(F.sum("spend").alias("amount"))
    .orderBy("platform", "billing_month")
    .collect())

invoices = []
for i, r in enumerate(delivered, start=1):
    inv = {"invoice_id": f"INV-{i:04d}", "vendor": r["platform"],
           "billing_month": r["billing_month"], "amount": r["amount"], "currency": "USD"}
    key = (r["platform"], r["billing_month"])

    if key == ("daily_campaign_platform", "2019-08"):
        inv["amount"] -= Decimal("1835.00")                                      # Plant 1
    elif key == ("multichannel_platform", "2021-03"):
        inv["amount"] = (inv["amount"] * Decimal("1.025")).quantize(Decimal("0.01"))  # Plant 2
    elif key == ("multichannel_platform", "2021-12"):
        continue                                                                  # Plant 3

    invoices.append(inv)
    if key == ("multichannel_platform", "2021-07"):
        invoices.append(dict(inv))                                                # Plant 4

invoices.append({"invoice_id": "INV-9001", "vendor": "multichannel_platform",
                 "billing_month": "2022-01", "amount": Decimal("15000.00"),
                 "currency": "USD"})                                              # Plant 5

target = f"/Volumes/workspace/media_pipeline/bronze/finance_billing/ingest_date={date.today().isoformat()}"
os.makedirs(target, exist_ok=True)
with open(f"{target}/billing.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["invoice_id", "vendor", "billing_month", "amount", "currency"])
    writer.writeheader()
    writer.writerows(invoices)

print(f"{len(invoices)} invoice lines written to {target}")